# CHAPTER 5 
## 5.3: Resampling Methods 

### Labs for cross-Validation and the Bootstrap Lab

In [3]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize,
poly)
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import \
(cross_validate,
KFold,
ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

# 5.3.1:Validation Set Approach 
### Using the function train_test_split() to split the data into training train_test_split() and validation sets.

In [4]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto,
                                          test_size=196,
                                          random_state=0)

### Fitting a linear regression using only the observations corresponding to the training set Auto_train

In [5]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

### Using the predict() method of results evaluated on the model matrix for this model created using the validation data set. We also calculatethe validation MSE of our model.

In [6]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((y_valid - valid_pred)**2)

np.float64(23.61661706966988)

### Estimating the validation error for higher-degree polynomial regressions.

In [7]:
def evalMSE(terms,
            response,
            train,
            test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]

    X_test = mm.transform(test)
    y_test = test[response]
   
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean((y_test - test_pred)**2)

### Estimating the validation MSE using linear, quadratic and cubic fits. We use the enumerate() function

In [8]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                        Auto_train,
                        Auto_valid)
MSE

array([23.61661707, 18.76303135, 18.79694163])

### If we choose a different training/validation split instead, then we can expect somewhat different errors on the validation set.

In [9]:
Auto_train, Auto_valid = train_test_split(Auto,
                                          test_size=196,
                                          random_state=3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

# 5.3.2 :Cross-Validation 
## Using sklearn to perform cross-validation

In [10]:
hp_model = sklearn_sm(sm.OLS,
                      MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,
                            X,
                            Y,
                            cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(24.23151351792922)

### Using cross_validate() function to produce a dictionary with several components

In [11]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.4244303 , 19.0332261 ])

### Introducing the outer() method of the np.power() function

In [12]:
A = np.array([3, 5, 9])
B = np.array([2, 4])
np.add.outer(A, B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

### use random_state to set a random seed and initialize a vector cv_error in which we will store the CV errors corresponding to the polynomial fits of degrees one to five.

In [13]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0) # use same splits for each degree
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13718682])

### Using the ShuffleSplit() funtion to implement the validation set approach just as easily as K-fold cross-validation.

In [14]:
validation = ShuffleSplit(n_splits=1,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation);
results['test_score']


array([23.61661707])

###  Estimating the variability in the test error.

In [15]:
validation = ShuffleSplit(n_splits=10,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation)
results['test_score'].mean(), results['test_score'].std()

(np.float64(23.802232661034164), np.float64(1.4218450941091831))

# 5.3.3 :The Bootstrap

# Estimating the Accuracy of a Statistic of Interest
### The Portfolio data set in the ISLP package to estimate the sampling variance of the parameter α

In [16]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X','Y']].loc[idx], rowvar=False)
    return ((cov_[1,1] - cov_[0,1]) /
            (cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

### The function alpha_func() returns an estimate for α based on applying the minimum variance formula

In [17]:
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

### Randomly select 100 observations from range(100), with replacement. This is equivalent to constructing a new bootstrap data set and recomputing αˆ based on the new data set.

In [18]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
           rng.choice(100,
                      100,
                      replace=True))

np.float64(0.6074452469619004)

### creating a simple function boot_SE() for computing the bootstrap standard error for arbitrary functions that take only a data frame as an argument

In [19]:
def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

### Using our function to evaluate the accuracy of our estimate of α using B = 1,000 bootstrap replications.

In [20]:
alpha_SE = boot_SE(alpha_func,
                   Portfolio,
                   B=1000,
                   seed=0)
alpha_SE

np.float64(0.09118176521277699)

# Estimating the Accuracy of a Linear Regression Model
### Using the clone() function to make a copy of the formula that can clone() be refit to the new dataframe.

In [21]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

### Using function, partial() to freeze the first two model-formula arguments of boot_OLS().

In [22]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

### Using hp_func() function to create bootstrap estimates for the intercept and slope terms by randomly sampling from among the observations with replacement.

In [23]:
rng = np.random.default_rng(0)

np.array([
    hp_func(
        Auto,
        rng.choice(Auto.index, size=len(Auto), replace=True)
    )
    for _ in range(10)
])


array([[39.12226577, -0.1555926 ],
       [37.18648613, -0.13915813],
       [37.46989244, -0.14112749],
       [38.56723252, -0.14830116],
       [38.95495707, -0.15315141],
       [39.12563927, -0.15261044],
       [38.45763251, -0.14767251],
       [38.43372587, -0.15019447],
       [37.87581142, -0.1409544 ],
       [37.95949036, -0.1451333 ]])

###  Using the boot_SE() function to compute the standard errors of 1,000 bootstrap estimates for the intercept and slope terms.

In [24]:
hp_se = boot_SE(hp_func,
                Auto,
                B=1000,
                seed=10)
hp_se

intercept     0.731176
horsepower    0.006092
dtype: float64

### Computing the standard errors for the regression coefficients in a linear model using the summarize() function from ISLP.sm.

In [25]:
hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

### Compute the bootstrap standard error estimates and the standard linear regression estimates that result from fitting the quadratic model to the data.

In [26]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS,
                    quad_model,
                    'mpg')
boot_SE(quad_func, Auto, B=1000)


intercept                                  1.538641
poly(horsepower, degree=2, raw=True)[0]    0.024696
poly(horsepower, degree=2, raw=True)[1]    0.000090
dtype: float64

# Here we compare the results to the standard errors computed using sm.OLS() 

In [27]:
M = sm.OLS(Auto['mpg'],
           quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64